In [0]:
silver_path = "abfss://silver@stmedallionmiguel.dfs.core.windows.net/sales"
df = spark.read.format("delta").load(silver_path)
df.printSchema()
print(df.count())
display(df.limit(10))

root
 |-- sales_order_number: string (nullable = true)
 |-- sales_order_line_number: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email_address: string (nullable = true)
 |-- item: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- tax_amount: double (nullable = true)

32718


sales_order_number,sales_order_line_number,order_date,customer_name,email_address,item,quantity,unit_price,tax_amount
SO43701,1,2019-07-01,Christy Zhu,christy12@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
SO43704,1,2019-07-01,Julio Ruiz,julio1@adventure-works.com,"Mountain-100 Black, 48",1,3374.99,269.9992
SO43705,1,2019-07-01,Curtis Lu,curtis9@adventure-works.com,"Mountain-100 Silver, 38",1,3399.99,271.9992
SO43700,1,2019-07-01,Ruben Prasad,ruben10@adventure-works.com,"Road-650 Black, 62",1,699.0982,55.9279
SO43703,1,2019-07-01,Albert Alvarez,albert7@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
SO43697,1,2019-07-01,Cole Watson,cole1@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
SO43699,1,2019-07-01,Sydney Wright,sydney61@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
SO43702,1,2019-07-01,Colin Anand,colin45@adventure-works.com,"Road-150 Red, 44",1,3578.27,286.2616
SO43698,1,2019-07-01,Rachael Martinez,rachael16@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
SO43707,1,2019-07-02,Emma Brown,emma3@adventure-works.com,"Road-150 Red, 48",1,3578.27,286.2616


In [0]:
from pyspark.sql import functions as F

df.select(
    F.min("order_date").alias("primeira_venda"),
    F.max("order_date").alias("ultima_venda"),
    F.countDistinct("sales_order_number").alias("pedidos"),
    F.countDistinct("customer_name").alias("clientes"),
    F.countDistinct("item").alias("produtos"),
    F.sum(F.col("quantity") * F.col("unit_price")).alias("receita_total"),
    F.sum((F.col("unit_price") <= 0).cast("int")).alias("precos_zerados"),
).show()

+--------------+------------+-------+--------+--------+--------------------+--------------+
|primeira_venda|ultima_venda|pedidos|clientes|produtos|       receita_total|precos_zerados|
+--------------+------------+-------+--------+--------+--------------------+--------------+
|    2019-07-01|  2021-12-31|  16459|   12387|     130|2.0928022430685084E7|             0|
+--------------+------------+-------+--------+--------+--------------------+--------------+



In [0]:
from pyspark.sql import functions as F

gold_base = "abfss://gold@stmedallionmiguel.dfs.core.windows.net"
vendas = df.withColumn("receita", F.round(F.col("quantity") * F.col("unit_price"), 2))
display(vendas.select("sales_order_number", "item", "quantity", "unit_price", "receita").limit(5))

sales_order_number,item,quantity,unit_price,receita
SO43701,"Mountain-100 Silver, 44",1,3399.99,3399.99
SO43704,"Mountain-100 Black, 48",1,3374.99,3374.99
SO43705,"Mountain-100 Silver, 38",1,3399.99,3399.99
SO43700,"Road-650 Black, 62",1,699.0982,699.1
SO43703,"Road-150 Red, 62",1,3578.27,3578.27


In [0]:
vendas = vendas.withColumn("ano", F.year("order_date")).withColumn("mes", F.month("order_date"))
display(vendas.select("order_date", "ano", "mes").limit(5))

order_date,ano,mes
2019-07-01,2019,7
2019-07-01,2019,7
2019-07-01,2019,7
2019-07-01,2019,7
2019-07-01,2019,7


In [0]:
vendas_mensais = vendas.groupBy("ano", "mes").agg(
    F.countDistinct("sales_order_number").alias("qtd_pedidos"),
    F.sum("quantity").alias("itens_vendidos"),
    F.round(F.sum("receita"), 2).alias("receita"),
    F.round(F.sum("tax_amount"), 2).alias("imposto"),
)
display(vendas_mensais.orderBy("ano", "mes"))

ano,mes,qtd_pedidos,itens_vendidos,receita,imposto
2019,7,289,289,934028.8,74722.3
2019,8,159,159,519494.17,41559.53
2019,9,161,161,513329.51,41066.36
2019,10,174,174,561681.51,44934.52
2019,11,230,230,737839.87,59027.19
2019,12,188,188,596746.6,47739.73
2020,1,193,193,614557.98,49164.64
2020,2,177,177,580393.99,46431.52
2020,3,219,219,700873.23,56069.86
2020,4,202,202,636375.35,50910.03


In [0]:
vendas_mensais = vendas_mensais.withColumn("ticket_medio", F.round(F.col("receita") / F.col("qtd_pedidos"), 2))
display(vendas_mensais.orderBy("ano", "mes"))
print(vendas_mensais.agg(F.sum("receita")).collect()[0][0])

ano,mes,qtd_pedidos,itens_vendidos,receita,imposto,ticket_medio
2019,7,289,289,934028.8,74722.3,3231.93
2019,8,159,159,519494.17,41559.53,3267.26
2019,9,161,161,513329.51,41066.36,3188.38
2019,10,174,174,561681.51,44934.52,3228.05
2019,11,230,230,737839.87,59027.19,3208.0
2019,12,188,188,596746.6,47739.73,3174.18
2020,1,193,193,614557.98,49164.64,3184.24
2020,2,177,177,580393.99,46431.52,3279.06
2020,3,219,219,700873.23,56069.86,3200.33
2020,4,202,202,636375.35,50910.03,3150.37


20928023.1


In [0]:
vendas_mensais.write.format("delta").mode("overwrite").save(f"{gold_base}/vendas_mensais")

In [0]:
check = spark.read.format("delta").load(f"{gold_base}/vendas_mensais")
print(check.count())
check.printSchema()

30
root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- qtd_pedidos: long (nullable = true)
 |-- itens_vendidos: long (nullable = true)
 |-- receita: double (nullable = true)
 |-- imposto: double (nullable = true)
 |-- ticket_medio: double (nullable = true)



In [0]:
vendas_por_produto = vendas.groupBy("item").agg(
    F.countDistinct("sales_order_number").alias("qtd_pedidos"),
    F.sum("quantity").alias("unidades_vendidas"),
    F.round(F.sum("receita"), 2).alias("receita"),
)
display(vendas_por_produto.orderBy(F.desc("receita")))

item,qtd_pedidos,unidades_vendidas,receita
"Road-150 Red, 48",337,337,1205876.99
"Road-150 Red, 62",336,336,1202298.72
"Road-150 Red, 52",302,302,1080637.54
"Road-150 Red, 56",295,295,1055589.65
"Road-150 Red, 44",281,281,1005493.87
"Mountain-200 Black, 46",425,425,925946.86
"Mountain-200 Black, 42",388,388,844474.69
"Mountain-200 Silver, 46",371,371,816222.26
"Mountain-200 Silver, 38",370,370,815145.12
"Mountain-200 Black, 38",366,366,799148.6


In [0]:
vendas_por_produto.write.format("delta").mode("overwrite").save(f"{gold_base}/vendas_por_produto")

In [0]:
df_check = spark.read.format("delta").load(f"{gold_base}/vendas_por_produto")
print(df_check.count())
df_check.printSchema()

130
root
 |-- item: string (nullable = true)
 |-- qtd_pedidos: long (nullable = true)
 |-- unidades_vendidas: long (nullable = true)
 |-- receita: double (nullable = true)



In [0]:
vendas_por_cliente = (vendas.groupBy("email_address").agg(
    F.first("customer_name").alias("cliente"),
    F.countDistinct("sales_order_number").alias("qtd_pedidos"),
    F.sum("quantity").alias("unidades_compradas"),
    F.round(F.sum("receita"), 2).alias("receita"),
))
display(vendas_por_cliente.orderBy(F.desc("receita")).limit(10))
print(vendas_por_cliente.count())

email_address,cliente,qtd_pedidos,unidades_compradas,receita
larry17@adventure-works.com,Larry Vazquez,4,11,10899.62
kaitlyn72@adventure-works.com,Kaitlyn Henderson,4,11,10896.22
nichole16@adventure-works.com,Nichole Nara,4,11,10876.32
kate19@adventure-works.com,Kate Anand,4,12,10872.06
margaret25@adventure-works.com,Margaret He,4,11,10841.22
lawrence6@adventure-works.com,Lawrence Alonso,4,11,10836.9
terrance17@adventure-works.com,Terrance Rodriguez,4,11,10829.22
rosa20@adventure-works.com,Rosa Hu,4,12,10817.6
aaron52@adventure-works.com,Aaron Wright,4,9,10813.63
clarence8@adventure-works.com,Clarence Gao,4,11,10799.52


12427


In [0]:
vendas_por_cliente.write.format("delta").mode("overwrite").save(f"{gold_base}/vendas_por_cliente")

In [0]:
df_check = spark.read.format("delta").load(f"{gold_base}/vendas_por_cliente")
print(df_check.count())
df_check.printSchema()

12427
root
 |-- email_address: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- qtd_pedidos: long (nullable = true)
 |-- unidades_compradas: long (nullable = true)
 |-- receita: double (nullable = true)

